# User Retention and Churn Analysis
## Notebook 2: Data Preprocessing & Feature Engineering

**Author:** Vijayalakshmi Veeraiyan  
**Goal:** Clean the raw data, engineer useful features, and prepare the dataset for machine learning modeling.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

print('Libraries loaded successfully!')

## 2. Load the Dataset

In [ ]:
file_path = '/content/telecom_customer_churn.csv'
df = pd.read_csv(file_path)
print(f'Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns')

## 3. Data Cleaning

Based on the exploration in Notebook 1, we identified:
- Negative values in monetary columns (e.g. -4 in Monthly Charge)
- Missing values in Churn Category and Churn Reason for non-churned customers
- Zip Code stored as integer (should be string to preserve leading zeros)

In [ ]:
def clean_data(df):
    """
    Clean the raw dataset:
    - Fix negative monetary values
    - Handle missing values in churn-related columns
    - Fix data types
    """
    df_clean = df.copy()

    # Fix negative values in monetary columns
    monetary_columns = [
        'Monthly Charge', 'Total Charges', 'Total Refunds',
        'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue'
    ]

    for col in monetary_columns:
        negative_count = (df_clean[col] < 0).sum()
        if negative_count > 0:
            print(f'Fixed {negative_count} negative values in {col}')
            df_clean[col] = np.where(
                df_clean[col] < 0,
                df_clean[col].median(),
                df_clean[col]
            )

    # Convert Zip Code to string
    df_clean['Zip Code'] = df_clean['Zip Code'].astype(str)

    # Fill missing Churn Category and Reason for non-churned customers
    df_clean['Churn Category'] = df_clean['Churn Category'].fillna('No Churn')
    df_clean['Churn Reason'] = df_clean['Churn Reason'].fillna('No Churn')

    print(f'\nCleaning complete. Dataset shape: {df_clean.shape}')
    return df_clean

df_clean = clean_data(df)
print('\nMissing values after cleaning:')
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

## 4. Feature Engineering

Creating new features based on domain knowledge and observations from the data exploration:
- **Tenure in years** — more interpretable than months
- **Age groups** — to capture generational patterns
- **Premium services count** — total number of add-ons per customer
- **High data user flag** — binary indicator for heavy data usage
- **Monthly spend ratio** — monthly charge relative to tenure (key churn signal)

In [ ]:
def feature_engineering(df):
    """
    Engineer new features from existing columns:
    - Binary churn target variable
    - Tenure in years
    - Age group bins
    - Premium services count
    - High data user flag
    - Monthly spend ratio
    """
    df_feat = df.copy()

    # Target variable — binary churn indicator
    df_feat['Churned_Binary'] = np.where(
        df_feat['Customer Status'] == 'Churned', 1, 0
    )

    # Tenure in years
    df_feat['Tenure_Years'] = df_feat['Tenure in Months'] / 12

    # Age groups
    df_feat['Age_Group'] = pd.cut(
        df_feat['Age'],
        bins=[0, 30, 45, 60, 100],
        labels=['Under 30', '30-45', '46-60', 'Over 60']
    )

    # Count of premium services per customer
    premium_services = [
        'Online Security', 'Online Backup', 'Device Protection Plan',
        'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music'
    ]
    df_feat['Premium_Services_Count'] = 0
    for service in premium_services:
        df_feat['Premium_Services_Count'] += np.where(
            df_feat[service] == 'Yes', 1, 0
        )

    # High data user flag (above 50 GB/month)
    df_feat['High_Data_User'] = np.where(
        df_feat['Avg Monthly GB Download'] > 50, 1, 0
    )

    # Monthly spend ratio — high ratio = paying a lot relative to tenure
    # This captures customers who are expensive but haven't been with us long
    df_feat['Monthly_Spend_Ratio'] = (
        df_feat['Monthly Charge'] /
        df_feat['Tenure in Months'].replace(0, 1)  # avoid division by zero
    )

    print('New features created:')
    new_features = ['Churned_Binary', 'Tenure_Years', 'Age_Group',
                    'Premium_Services_Count', 'High_Data_User', 'Monthly_Spend_Ratio']
    for f in new_features:
        print(f'  - {f}')

    return df_feat

df_featured = feature_engineering(df_clean)
print(f'\nDataset shape after feature engineering: {df_featured.shape}')

## 5. Visualise Engineered Features

In [ ]:
# Monthly Spend Ratio — churned vs stayed
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly Spend Ratio distribution
churned_ratio = df_featured[df_featured['Churned_Binary'] == 1]['Monthly_Spend_Ratio']
stayed_ratio = df_featured[df_featured['Churned_Binary'] == 0]['Monthly_Spend_Ratio']

axes[0].hist(stayed_ratio, bins=40, alpha=0.6, label='Stayed', color='steelblue')
axes[0].hist(churned_ratio, bins=40, alpha=0.6, label='Churned', color='salmon')
axes[0].set_title('Monthly Spend Ratio: Churned vs Stayed', fontsize=12)
axes[0].set_xlabel('Monthly Spend Ratio')
axes[0].set_ylabel('Count')
axes[0].legend()

# Premium Services Count by churn status
premium_churn = df_featured.groupby('Premium_Services_Count')['Churned_Binary'].mean() * 100
axes[1].bar(premium_churn.index, premium_churn.values, color='steelblue')
axes[1].set_title('Churn Rate by Premium Services Count (%)', fontsize=12)
axes[1].set_xlabel('Number of Premium Services')
axes[1].set_ylabel('Churn Rate (%)')

plt.tight_layout()
plt.savefig('engineered_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Prepare Data for Modeling

Steps:
- Define target variable (Churned_Binary)
- Remove columns not needed for modeling
- Separate categorical and numerical features
- Build preprocessing pipelines
- Split into train and test sets

In [ ]:
def prepare_modeling_data(df):
    """
    Prepare data for modeling:
    - Define X (features) and y (target)
    - Build preprocessing pipelines for numerical and categorical features
    - Split into train and test sets (80/20)
    """
    df_prep = df.copy()

    # Target variable
    y = df_prep['Churned_Binary']

    # Drop columns not needed for modeling
    columns_to_drop = [
        'Customer ID', 'Churned_Binary', 'Customer Status',
        'Churn Category', 'Churn Reason',
        'Zip Code', 'Latitude', 'Longitude', 'City'
    ]
    X = df_prep.drop(columns=columns_to_drop)

    # Identify categorical and numerical columns
    categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

    print(f'Features: {X.shape[1]} total')
    print(f'  Categorical: {len(categorical_cols)}')
    print(f'  Numerical: {len(numerical_cols)}')

    # Preprocessing pipeline for numerical features
    numerical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),  # fill missing with median
        ('scaler', StandardScaler())                    # scale to mean=0, std=1
    ])

    # Preprocessing pipeline for categorical features
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),  # fill missing with mode
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # Combine both pipelines
    preprocessor = ColumnTransformer(transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

    # Train/test split — 80% train, 20% test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Apply preprocessing
    X_train_processed = preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.transform(X_test)

    # Get feature names after one-hot encoding
    cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
    cat_feature_names = cat_encoder.get_feature_names_out(categorical_cols).tolist()
    all_feature_names = numerical_cols + cat_feature_names

    print(f'\nTrain set: {X_train_processed.shape[0]} samples')
    print(f'Test set: {X_test_processed.shape[0]} samples')
    print(f'Total features after encoding: {X_train_processed.shape[1]}')

    return (
        X_train_processed, X_test_processed,
        y_train, y_test,
        preprocessor, all_feature_names
    )

X_train, X_test, y_train, y_test, preprocessor, feature_names = prepare_modeling_data(df_featured)

print(f'\nClass distribution in training set:')
print(pd.Series(y_train).value_counts(normalize=True).round(3))

## 7. Save Processed Data

Save the processed arrays for use in Notebook 3.

In [ ]:
import joblib
import numpy as np

# Save processed data
np.save('/content/X_train.npy', X_train)
np.save('/content/X_test.npy', X_test)
np.save('/content/y_train.npy', y_train)
np.save('/content/y_test.npy', y_test)

# Save preprocessor and feature names
joblib.dump(preprocessor, '/content/preprocessor.pkl')
joblib.dump(feature_names, '/content/feature_names.pkl')

print('All processed data saved successfully!')
print('Files saved:')
print('  X_train.npy, X_test.npy, y_train.npy, y_test.npy')
print('  preprocessor.pkl, feature_names.pkl')
print('\nOpen Notebook 3 (3_modeling.ipynb) to train and evaluate the model.')

## 8. Preprocessing Summary

| Step | Action | Reason |
|---|---|---|
| Negative values | Replaced with median | Data quality issue |
| Missing churn columns | Filled with 'No Churn' | Customers who stayed have no churn reason |
| Zip Code | Converted to string | Preserve leading zeros |
| Numerical missing | Imputed with median | Robust to outliers |
| Categorical missing | Imputed with mode | Most common value |
| Numerical scaling | StandardScaler | Required for gradient-based models |
| Categorical encoding | OneHotEncoder | Convert text to numbers |
| Train/test split | 80/20 | Standard evaluation practice |

---
**Next:** Open `3_modeling.ipynb` to train and evaluate machine learning models.